# Download tabletop backgrounds

Choose `poly_haven` for relevant wood, fabric, carpet, leather and concrete surface textures, or `commons` for photo searches. Poly Haven uses four concurrent downloads and requires no account. Powered by [Poly Haven](https://polyhaven.com/); textures are CC0.

Set N to the target number of unique sources. The texture collection may contain fewer matching sources than N. Review downloaded surfaces below before generation uses them. Each source has a fixed train/validation partition and a manifest with attribution. Re-running resumes downloads.


In [1]:
from pathlib import Path
import html
import math
import cv2
import ipywidgets as widgets
from IPython.display import display

from set_master.generation.backgrounds import (
    DEFAULT_QUERIES,
    download_backgrounds,
    download_tabletop_textures,
    load_backgrounds,
    save_backgrounds,
)
from set_master.paths import BACKGROUNDS_DIR, TABLETOP_TEXTURES_DIR

source = "poly_haven"  # poly_haven or commons
N = 100
background_dir = TABLETOP_TEXTURES_DIR if source == "poly_haven" else BACKGROUNDS_DIR
queries = DEFAULT_QUERIES
min_side = 480
max_results_per_query = 500
user_agent = "SetMasterBackgroundDownloader/1.0 (tabletop research notebook)"

In [2]:
progress = widgets.IntProgress(value=0, max=N, description="Downloads")
display(progress)


def report(done, total):
    progress.value = min(done, total)


if source == "poly_haven":
    records = download_tabletop_textures(count=N, root=background_dir, progress=report)
elif source == "commons":
    records = download_backgrounds(
        count=N,
        root=background_dir,
        queries=queries,
        user_agent=user_agent,
        min_side=min_side,
        max_results_per_query=max_results_per_query,
        progress=report,
    )
else:
    raise ValueError("Select poly_haven or commons")
print(f"{len(records)} sources downloaded")
progress.value = min(len(records), N)

IntProgress(value=0, description='Downloads', max=1000)

KeyboardInterrupt: 

## Review usable surfaces

Approve images with a usable tabletop or surface where cards could plausibly lie. Skip card/game photos, people, collages, and views dominated by furniture or objects. Some realistic distractors are useful, provided they don't fill the placement surface. The generator currently places cards anywhere in the image, so approve surface textures or crop photos to the usable area before approval.

Use the checkboxes and **Save page**. Approval defaults to off. Each source belongs to a fixed training or validation partition; both source files and license metadata are retained.


In [ ]:
records = load_backgrounds(background_dir)
page_size = 12
page = widgets.BoundedIntText(value=1, min=1, max=max(1, math.ceil(len(records) / page_size)), description="Page")
save_button = widgets.Button(description="Save page", button_style="primary")
gallery = widgets.VBox()
review_status = widgets.HTML()
choices = {}


def show_page(change=None):
    choices.clear()
    tiles = []
    start = (page.value - 1) * page_size
    for record in records[start : start + page_size]:
        image = cv2.imread(str(background_dir / record.file_name))
        if image is None:
            raise FileNotFoundError(record.file_name)
        height, width = image.shape[:2]
        scale = min(320 / width, 220 / height)
        thumbnail = cv2.resize(image, (round(width * scale), round(height * scale)))
        success, encoded = cv2.imencode(".jpg", thumbnail)
        if not success:
            raise OSError(f"Could not preview {record.file_name}")
        checkbox = widgets.Checkbox(value=record.approved, description="Use this background", indent=False)
        choices[record.source_id] = checkbox
        link = widgets.HTML(
            f'<a href="{html.escape(record.source_url, quote=True)}" target="_blank">{html.escape(Path(record.file_name).name)}</a> · {html.escape(record.split)} · {html.escape(record.license)}'
        )
        tiles.append(widgets.VBox([widgets.Image(value=encoded.tobytes(), format="jpg", width=320), link, checkbox]))
    gallery.children = tuple(widgets.HBox(tiles[i : i + 3]) for i in range(0, len(tiles), 3))
    review_status.value = f"{sum(r.approved for r in records)} / {len(records)} approved. Save before changing pages."


def save_page(button):
    for record in records:
        if record.source_id in choices:
            record.approved = choices[record.source_id].value
    save_backgrounds(background_dir, records, source="Poly Haven" if source == "poly_haven" else "Wikimedia Commons")
    review_status.value = f"Saved: {sum(r.approved for r in records)} approved."


page.observe(show_page, names="value")
save_button.on_click(save_page)
show_page()
display(widgets.VBox([widgets.HBox([page, save_button]), review_status, gallery]))

## Generate card scenes

Open `generate_synthetic_dataset.ipynb` and select **real_scene** or **mixed** under **Backgrounds**. Only approved images from the selected partition are used. Gray remains the default.

Generate separate training and validation datasets using the `train` and `validation` background partitions, then set `val_data` to the validation dataset in the training notebook. This keeps the same source backgrounds out of both datasets. The split separates source images, not necessarily photos of the same physical table; review related photos accordingly.

Retain `manifest.json` and the generated per-image background metadata when sharing a dataset so source attribution remains available.
